# Configuración SSH para GitHub

Este notebook genera un resumen parametrizado de los comandos. Para conocer el propósito, las advertencias y la explicación de cada paso, consulta [`doc.md`](./doc.md).

> Los comandos generados están pensados para **Git Bash en Windows**. Revísalos antes de copiarlos y ejecutarlos.

In [41]:
# Cuenta y repositorio
account_email = "ID+USERNAME@users.noreply.github.com"
account_name = "foo"
repo_owner = "Oof"
repo_name = "repository"

# Clave SSH: id_<tipo>_<servicio>_<cuenta>_<dispositivo>
key_type = "ed25519"
service = "github"
device = "laptopFoo"
ssh_name = f"id_{key_type}_{service}_{account_name}_{device}"

# Alias: <servicio>-<cuenta_abreviada>-<dispositivo_abreviado>
account_abbreviation = "f"
device_abbreviation = "LF"
host_alias = f"{service}-{account_abbreviation}-{device_abbreviation}"
host_name = "github.com"

private_key = f"~/.ssh/{ssh_name}"
public_key = f"{private_key}.pub"
remote_url = f"git@{host_alias}:{repo_owner}/{repo_name}.git"

In [42]:
print(f"""
# PASO 1 — Crear la clave
ssh-keygen -t {key_type} -C "{account_email}"

# Cuando se solicite la ruta, usar este nombre:
{private_key}

# PASO 2 — Copiar la clave pública
cat {public_key} | clip

# Añadirla en GitHub → Settings → SSH and GPG keys como Authentication key.
# Para firmar commits, subir el mismo texto otra vez como Signing key.
""")


# PASO 1 — Crear la clave
ssh-keygen -t ed25519 -C "ID+USERNAME@users.noreply.github.com"

# Cuando se solicite la ruta, usar este nombre:
~/.ssh/id_ed25519_github_foo_laptopFoo

# PASO 2 — Copiar la clave pública
cat ~/.ssh/id_ed25519_github_foo_laptopFoo.pub | clip

# Añadirla en GitHub → Settings → SSH and GPG keys como Authentication key.
# Para firmar commits, subir el mismo texto otra vez como Signing key.



In [43]:
print(f"""
# PASO 3 — Añadir el bloque a ~/.ssh/config
cd ~/.ssh
nano config

Host {host_alias}
    HostName {host_name}
    User git
    IdentityFile {private_key}
    IdentitiesOnly yes

# Probar la conexión (SSH solicitará la passphrase si la clave no está en un agente):
ssh -T git@{host_alias}
""")


# PASO 3 — Añadir el bloque a ~/.ssh/config
cd ~/.ssh
nano config

Host github-f-LF
    HostName github.com
    User git
    IdentityFile ~/.ssh/id_ed25519_github_foo_laptopFoo
    IdentitiesOnly yes

# Probar la conexión (SSH solicitará la passphrase si la clave no está en un agente):
ssh -T git@github-f-LF



In [44]:
print(f"""
# PASOS 4 Y 5 — Configurar el repositorio local
git remote set-url origin {remote_url}
git remote -v

git config user.name "{account_name}"
git config user.email "{account_email}"
""")


# PASOS 4 Y 5 — Configurar el repositorio local
git remote set-url origin git@github-f-LF:Oof/repository.git
git remote -v

git config user.name "foo"
git config user.email "ID+USERNAME@users.noreply.github.com"



## Paso 6 — Firma SSH opcional

GitHub solo necesita que la clave pública se haya añadido como **Signing key**. El archivo `allowed_signers` se configura a continuación únicamente para verificar firmas de forma local.

In [45]:
print(f"""
# GIT BASH — Configuración local de allowed_signers
touch ~/.ssh/allowed_signers
git config --global gpg.ssh.allowedSignersFile ~/.ssh/allowed_signers

key_data="$(awk '{{print $2}}' {public_key})"
if ! grep -Fq "$key_data" ~/.ssh/allowed_signers; then
    awk -v principal="{account_email}" -v comment="{device}" '{{print principal, $1, $2, comment}}' {public_key} >> ~/.ssh/allowed_signers
fi

# REPOSITORIO LOCAL — Activar la firma
git config gpg.format ssh
git config user.signingkey {public_key}
git config commit.gpgsign true
""")


# GIT BASH — Configuración local de allowed_signers
touch ~/.ssh/allowed_signers
git config --global gpg.ssh.allowedSignersFile ~/.ssh/allowed_signers

key_data="$(awk '{print $2}' ~/.ssh/id_ed25519_github_foo_laptopFoo.pub)"
if ! grep -Fq "$key_data" ~/.ssh/allowed_signers; then
    awk -v principal="ID+USERNAME@users.noreply.github.com" -v comment="laptopFoo" '{print principal, $1, $2, comment}' ~/.ssh/id_ed25519_github_foo_laptopFoo.pub >> ~/.ssh/allowed_signers
fi

# REPOSITORIO LOCAL — Activar la firma
git config gpg.format ssh
git config user.signingkey ~/.ssh/id_ed25519_github_foo_laptopFoo.pub
git config commit.gpgsign true



In [46]:
print("""
# Verificar la firma
git commit --allow-empty -m "test ssh signing"
git log --show-signature -1
git push
""")


# Verificar la firma
git commit --allow-empty -m "test ssh signing"
git log --show-signature -1
git push



In [47]:
print("""
# PASO 7 — Introducir la passphrase una vez por sesión
pwd | clip
# Copiar la ruta del repositorio.

cd ~/.ssh
nano config
# Copiar el IdentityFile correspondiente a la cuenta.

cd "<ruta_del_repositorio_copiada>"
# Pegar la ruta con Windows + V.

eval "$(ssh-agent -s)"
ssh-add IdentityFile_copiado
# Sustituir IdentityFile_copiado por el valor copiado desde config.

# Extra: Comprobación silenciosa para grabaciones:
ssh-add -l > /dev/null 2>&1
case $? in
    0) echo "Agente activo con al menos una clave cargada." ;;
    1) echo "Agente accesible, pero no se listaron claves." ;;
    2) echo "No se pudo contactar al agente." ;;
esac
""")


# PASO 7 — Introducir la passphrase una vez por sesión
pwd | clip
# Copiar la ruta del repositorio.

cd ~/.ssh
nano config
# Copiar el IdentityFile correspondiente a la cuenta.

cd "<ruta_del_repositorio_copiada>"
# Pegar la ruta con Windows + V.

eval "$(ssh-agent -s)"
ssh-add IdentityFile_copiado
# Sustituir IdentityFile_copiado por el valor copiado desde config.

# Extra: Comprobación silenciosa para grabaciones:
ssh-add -l > /dev/null 2>&1
case $? in
    0) echo "Agente activo con al menos una clave cargada." ;;
    1) echo "Agente accesible, pero no se listaron claves." ;;
    2) echo "No se pudo contactar al agente." ;;
esac



In [48]:
print("""
# PASO 8 — Alias opcional de Git
git config --global alias.acp '!git add . && git commit && git push'

# Revisar antes de usar git acp:
git status

# Verificar o eliminar el alias:
git config --get-regexp '^alias\\.'
git config --global --unset alias.acp
""")


# PASO 8 — Alias opcional de Git
git config --global alias.acp '!git add . && git commit && git push'

# Revisar antes de usar git acp:
git status

# Verificar o eliminar el alias:
git config --get-regexp '^alias\.'
git config --global --unset alias.acp



# Usar una cuenta SSH ya configurada en un repositorio nuevo

In [49]:
print(f"""
# PASO 7 — Introducir la passphrase una vez por sesión
pwd | clip
# Copiar la ruta del repositorio.

cd ~/.ssh
nano config
# Copiar el Alias y IdentityFile correspondiente a la cuenta.
# Parte del IdentityFile es el SSH name.

cd "<ruta_del_repositorio_copiada>"
# Pegar la ruta con Windows + V.
""")


# PASO 7 — Introducir la passphrase una vez por sesión
pwd | clip
# Copiar la ruta del repositorio.

cd ~/.ssh
nano config
# Copiar el Alias y IdentityFile correspondiente a la cuenta.
# Parte del IdentityFile es el SSH name.

cd "<ruta_del_repositorio_copiada>"
# Pegar la ruta con Windows + V.



In [50]:
# Sustituir estos valores por los de la cuenta y el repositorio existentes.
existing_host_alias = "github-f-LF"
existing_ssh_name = "id_ed25519_github_foo_laptopFoo"
existing_account_name = "foo"
existing_account_email = "ID+USERNAME@users.noreply.github.com"
new_repo_owner = "foo"
new_repo_name = "new-repository"

print(f"""
eval "$(ssh-agent -s)"
ssh-add IdentityFile_copiado
# Sustituir IdentityFile_copiado por el valor copiado desde config.

git remote set-url origin git@{existing_host_alias}:{new_repo_owner}/{new_repo_name}.git
git remote -v

git config user.name "{existing_account_name}"
git config user.email "{existing_account_email}"

# Si se firmarán los commits en este repositorio:
git config gpg.format ssh
git config user.signingkey ~/.ssh/{existing_ssh_name}.pub
git config commit.gpgsign true
""")


eval "$(ssh-agent -s)"
ssh-add IdentityFile_copiado
# Sustituir IdentityFile_copiado por el valor copiado desde config.

git remote set-url origin git@github-f-LF:foo/new-repository.git
git remote -v

git config user.name "foo"
git config user.email "ID+USERNAME@users.noreply.github.com"

# Si se firmarán los commits en este repositorio:
git config gpg.format ssh
git config user.signingkey ~/.ssh/id_ed25519_github_foo_laptopFoo.pub
git config commit.gpgsign true

